# Why chain of thought helps a transformer

*ai · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/chain_of_thought)

The parity of $n$ bits is a running XOR, an inherently serial task. Identical tiny transformers (hand-written einsum attention, all models stacked on one model axis and trained in one run) learn it in two formats: answering at once, "$b_1 \dots b_n = p$", or writing the running parities first, "$b_1 \dots b_n : s_1 \dots s_n = p$". With the same weights and steps, the scratchpad models answer 98–100% of the test strings at $n = 4$ to $10$, while the direct models only fit $n = 4$ and are at or below chance beyond; the attention shows the two-token rule $s_k = s_{k-1} \oplus b_k$. A toy of the mechanism, not of how large language models reason.

**Vector form**

$$s_k = s_{k-1} \oplus b_k,\quad s_0 = 0,\quad p = s_n \qquad \mathbf A^{(m,h)} = \operatorname{softmax}\!\Big(\frac{\mathbf Q^{(m,h)}\mathbf K^{(m,h)\mathsf T}}{\sqrt{d_k}} + \mathbf M\Big) \qquad \mathcal L = \sum_m \mathcal L_m$$

**Index (tensor) form**

$$s_{nk} = \Big(\sum_{l \le k} b_{nl}\Big) \bmod 2 \qquad Q_{mnhik} = \hat X_{mnic} W^Q_{mhck} \qquad A_{mnhij} = \mathrm{softmax}_j\big(Q_{mnhik}K_{mnhjk}/\sqrt{d_k} + M_{ij}\big) \qquad \mathcal L_m = \frac{\sum_{n,i} \mu_{mni}\,\ell_{mni}}{\sum_{n,i}\mu_{mni}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $s_{nk}$ | `cumsum(bits, -1) % 2` | (N, n) | running parity of every prefix: the scratchpad targets, and $p = s_n$ |
| $t_{mni}$ | `make_batch(bits, n, cot, ctx)` | (M, N, T) | strings in the direct or the CoT format as token ids $0, 1, :, =, \_$; padded with blanks so every model of one $n$ shares one tensor |
| $X_{mnic}$ | `w['tok'][m, tokens] + w['pos'][:, None, :T]` | (M, N, T, d) | token plus learned position embedding, each model $m$ with its own tables |
| $Q_{mnhik}$ | `einsum('mnic,mhck->mnhik', X, Wq)` | (M, N, H, T, d_k) | repeated $c$ ⇒ summed; the model index $m$ is not summed, so $M$ independent transformers run in one product |
| $A_{mnhij}$ | `softmax(einsum('...ik,...jk->...ij', Q, K) / sqrt(dk) + M, -1)` | (M, N, H, T, T) | causal attention: each row is a probability distribution over the positions $j \le i$ |
| $\mu_{mni}$ | `mask` | (M, N, T) | which next tokens are trained: $p$ alone (direct), or $s_1 \dots s_n$, "=" and $p$ (CoT) |
| $\mathcal L_m$ | `masked_loss(logits, targets, mask)` | (M,) | mean $-\log p$ of the trained tokens of model $m$ (nats); their sum is minimised, and each term only reaches its own weights |
| $s_k \text{ written}$ | `greedy(w, bits, n, cot, ctx)` | (M, N, T) | at test time the model writes its own scratchpad, one token per forward pass, and answers after it |

Status: the effect shown is established. Parity is the textbook hard case for a transformer of fixed depth that must answer at once. Li, Liu, Zhou and Ma (ICLR 2024, arXiv 2402.12875) prove that constant-depth transformers without intermediate steps are limited to shallow parallel circuits ($\mathrm{TC}^0$-like, under their precision assumptions), while $T$ steps of chain of thought let them simulate a serial circuit of size $T$. Nye et al. (2021, arXiv 2112.00114) trained transformers to write a "scratchpad" of intermediate results and saw them solve multi-step tasks such as long addition and program tracing; Wei et al. (2022, arXiv 2201.11903) named chain-of-thought prompting for large language models. Method: a string of $n$ bits is written either directly, "$b_1 \dots b_n = p$", or with its running parities, "$b_1 \dots b_n : s_1 \dots s_n = p$". Every model is the same 2-layer, $d = 64$, 4-head decoder (the `tiny_transformer` code, with a model axis $m$ added to every einsum), trained with AdamW for the same number of steps and the same batch of strings; the CoT models read about twice as many tokens per string and are trained on their intermediate tokens, which is exactly the extra help a scratchpad gives. All models, $n$ by format by seed, plus the length-generalisation models, train in one run: their weights are stacked on the leading axis and gradient clipping is done per model, so they never influence one another (a test checks it). Data: for $n \ge 8$ a quarter of the $2^n$ strings (at most 1024) is held out and never trained on; for $n = 4$ and $6$ there are too few strings to hold any out, so those accuracies (starred on the plot) measure fitting, not generalisation. At test time a CoT model writes its own scratchpad greedily, token by token, and only its final answer is scored. Verdict: in the default run (2 seeds, numbers printed in the terminal), (a) the CoT models answer 0.98 to 1.00 of the test strings correctly at $n = 4, 6, 8, 10$; the direct models fit the 16 strings of $n = 4$ (1.00) but score 0.53, 0.27 and 0.53 at $n = 6, 8, 10$. A plausible reading of the below-chance score at $n = 8$ (not tested here): a held-out string differs from its training neighbours in one bit and so has the opposite parity, and a model that interpolates between neighbours gets it wrong. (b) Length generalisation fails for both: models trained on $1 \le n \le 8$ score 0.46 to 0.51 at $n = 12$ and $16$, because positions beyond 19 were never trained and, with the CoT layout used here, the bit $b_k$ sits $n + 1$ places before the cell that needs it, an offset that changes with $n$; the mixed-length CoT models master $n = 4$ (1.00) but not even $n = 8$ (0.56) in the step budget. (c) Blanking out the scratchpad after $k$ steps drops the answer to chance for every $k < n$: the answer is copied from the last cell, so all of the computation lives in the written tokens. (d) In the CoT model at $n = 8$, at every step $k$ some head puts 0.5 to 0.8 of the attention of the query that writes $s_k$ on $b_k$, against 0.08 for uniform attention (one head alone averages 0.48 over the steps; the heads share the steps between them). The other input of the rule, $s_{k-1}$, is the query token itself and reaches the output through the residual stream, so the learned rule is local: two tokens per step. Limits: this is a toy that shows the mechanism, not how large language models reason. It says nothing about whether the written steps of a large model are faithful to the computation that actually produces its answer; that is an open research question. Results vary between seeds and with the step budget, which is why every seed is plotted. Try more steps, $n$ up to 16, or a single layer.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/chain_of_thought/main.py
"""Main loop: parity strings → direct and chain-of-thought formats → M stacked transformers trained at once →
accuracy against n, length generalisation, hidden scratchpad steps and attention maps → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def _mean_pm(x: torch.Tensor) -> str:
    """'mean ± half-range' over the seeds (last axis)."""
    return f"{x.mean():.2f} ± {(x.max() - x.min()) / 2:.2f}"


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    lengths = [int(v) for v in str(params["lengths"]).split(",")]
    cfg = solve.Config(
        lengths=[4, 6] if fast else lengths,
        seeds=1 if fast else params["seeds"],
        steps=20 if fast else params["steps"],
        d=params["d_model"],
        heads=params["heads"],
        layers=params["layers"],
        batch=params["batch"],
        lr=params["lr"],
        gen_lengths=[4, 8] if fast else [4, 8, 12, 16],
        test_size=32 if fast else 256,
    )
    L, S = len(cfg.lengths), cfg.seeds
    t0 = time.time()
    res = solve.run_experiments(cfg, dev, n_checkpoints=4 if fast else 20)
    print(f"{res['M']} transformers trained together ({L} lengths × direct/CoT × {S} seeds + {2 * S} for length "
          f"generalisation), {cfg.layers} layers, d = {cfg.d}, {cfg.heads} heads, {cfg.steps} steps of "
          f"{cfg.batch} strings: {time.time() - t0:.0f} s")  # fmt: skip
    print(f"weights per model: {res['params_per_model'][0]} (n = {cfg.lengths[0]}) … {res['params_per_model'][-1]}; "
          "the direct and the CoT model of each n have the same count")  # fmt: skip

    print("(a) answer accuracy on test strings (chance 0.50), mean ± half-range over seeds:")
    for li, n in enumerate(cfg.lengths):
        note = "held-out strings" if res["held_out"][li] else f"all {2**n} strings, seen in training"
        print(f"  n = {n:2d}: direct {_mean_pm(res['acc'][li, 0])}   CoT {_mean_pm(res['acc'][li, 1])}   ({note})")
    print(
        "    with the true chain fed in, the CoT answer copies s_n in a fraction "
        f"{res['oracle_answer_agrees'].min():.3f} – {res['oracle_answer_agrees'].max():.3f} of cases"
    )
    if "gen_acc" in res:
        print(f"(b) trained on 1 ≤ n ≤ {cfg.train_max}, tested on longer strings:")
        for li, n in enumerate(cfg.gen_lengths):
            g = res["gen_acc"][:, :, li]
            print(f"  n = {n:2d}: direct {_mean_pm(g[0])}   CoT {_mean_pm(g[1])}")
    print("(c) CoT answer accuracy when only the first k scratchpad steps are kept (mean over seeds):")
    for li, n in enumerate(cfg.lengths):
        row = res["hidden"][li, :, : n + 1].mean(0)
        print(f"  n = {n:2d}: k = 0 … {n}: " + " ".join(f"{a:.2f}" for a in row.tolist()))
    l_best, h_best = res["best_head"]
    ns = res["n_show"]
    uniform = sum(1 / (ns + k) for k in range(1, ns + 1)) / ns  # query n + k − 1 sees n + k positions
    print(f"(d) CoT model, n = {res['n_show']}: layer {l_best + 1} head {h_best + 1} puts "
          f"{res['on_b'][l_best, h_best]:.2f} of the attention of the query that writes s_k on b_k "
          f"(uniform over the past would give {uniform:.2f}) and "
          f"{res['on_prev'][l_best, h_best]:.2f} on s_(k-1) itself")  # fmt: skip
    per_k = res["on_b_k"].flatten(0, 1).max(0).values  # (n,) best head at each step
    print("    best head at each step k = 1 … n, weight on b_k: " + " ".join(f"{a:.2f}" for a in per_k.tolist()))
    tok = res["sample_tok"].tolist()[: 2 * res["n_show"] + 2]  # bits, ":", the model-written chain, "="
    sample = "".join(tensors.VOCAB[t] for t in tok) + str(res["sample_answer"])
    print(f"    sample: {sample}  (model answer {res['sample_answer']}, true parity {res['sample_parity']})")

    plot.experiments_png(res, out_dir / "experiments.png")
    plot.write_table(res, out_dir / "accuracy.csv")
    return plot.animate(res, out_dir / "chain_of_thought.gif", hold=1 if fast else 2)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/chain_of_thought/equation.py
r"""Parity and M stacked transformers → tensor translation, every product an einsum over the model axis m.

Indices: m = model (format × length × seed), n = sequence, i = query position, j = key position, c = channel,
h = head, k = channel within a head, v = vocabulary token.

    s_k = b_1 \oplus \dots \oplus b_k = \big(\textstyle\sum_{l \le k} b_l\big) \bmod 2      running parity (the task)
    X_{mnic} = E_{m,t_{mni},c} + P_{mic}                                    token + position embedding
    Q_{mnhik} = \hat X_{mnic} W^Q_{mhck} + b^Q_{mhk}                        (c summed; K and V alike)
    A_{mnhij} = \mathrm{softmax}_j\big(Q_{mnhik}K_{mnhjk}/\sqrt{d_k} + M_{ij}\big)  (k summed)  M_{ij} = −∞ for j > i
    X_{mnic} \mathrel{+}= A_{mnhij} V_{mnhjk} W^O_{mhkc} + b^O_{mc}         (h, j, k summed)
    X \mathrel{+}= \mathrm{GELU}(\hat X W_1 + c_1) W_2 + c_2                MLP, \hat X = LayerNorm(X)
    \mathcal L_m = \sum_{n,i} \mu_{mni}\,(-\log \mathrm{softmax}_v z_{mniv}[y_{mni}]) / \sum_{n,i} \mu_{mni}

Each model m only ever multiplies its own weights, so summing the M losses gives every model its own gradient:
M independent trainings in one set of tensor ops.
"""

import math

import torch


def running_parity(bits: torch.Tensor) -> torch.Tensor:
    """s_k = b_1 ⊕ … ⊕ b_k for every prefix: the cumulative sum mod 2.  bits (..., n) → s (..., n)."""
    return torch.cumsum(bits, dim=-1) % 2


def layer_norm(X: torch.Tensor, g: torch.Tensor, b: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    """(X − mean) / std over the channels of every token, then a learned scale and shift.  X (M, N, T, d), g (M, d)."""
    mu = X.mean(-1, keepdim=True)
    var = (X - mu).pow(2).mean(-1, keepdim=True)  # biased variance over the channels
    return g[:, None, None] * (X - mu) / torch.sqrt(var + eps) + b[:, None, None]


def causal_mask(t: int, device: torch.device) -> torch.Tensor:
    """M_{ij} = 0 for j ≤ i (past and present) and −∞ for j > i (future).  (T, T)"""
    future = torch.triu(torch.ones(t, t, dtype=torch.bool, device=device), diagonal=1)
    return torch.zeros(t, t, device=device).masked_fill(future, float("-inf"))


def attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Causal scaled dot-product attention over leading axes.  Q, K, V (..., T, d_k) → out (..., T, d_k), A (..., T, T).

    S_{ij} = Q_{ik} K_{jk} / √d_k + M_{ij} (k summed); A = softmax_j S; out_{ik} = A_{ij} V_{jk} (j summed).
    """
    t = Q.shape[-2]
    S = torch.einsum("...ik,...jk->...ij", Q, K) / math.sqrt(Q.shape[-1])
    A = torch.softmax(S + causal_mask(t, Q.device).to(S.dtype), dim=-1)
    return torch.einsum("...ij,...jk->...ik", A, V), A


def self_attention(X: torch.Tensor, p: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """Multi-head causal self-attention of normalised tokens X (M, N, T, d) → update (M, N, T, d), A (M, N, H, T, T)."""
    Q = torch.einsum("mnic,mhck->mnhik", X, p["Wq"]) + p["bq"][:, None, :, None, :]  # c summed, per model m
    K = torch.einsum("mnjc,mhck->mnhjk", X, p["Wk"]) + p["bk"][:, None, :, None, :]
    V = torch.einsum("mnjc,mhck->mnhjk", X, p["Wv"]) + p["bv"][:, None, :, None, :]
    mix, A = attention(Q, K, V)
    return torch.einsum("mnhik,mhkc->mnic", mix, p["Wo"]) + p["bo"][:, None, None], A  # h, k summed


def mlp(X: torch.Tensor, p: dict) -> torch.Tensor:
    """Position-wise d → 4d → d network with a GELU, separate weights for every model m."""
    hidden = torch.nn.functional.gelu(torch.einsum("mnic,mcf->mnif", X, p["W1"]) + p["c1"][:, None, None])
    return torch.einsum("mnif,mfc->mnic", hidden, p["W2"]) + p["c2"][:, None, None]


def forward(tokens: torch.Tensor, w: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """tokens (M, N, T) → next-token logits z (M, N, T, V) and attention maps (M, N, layers, H, T, T)."""
    m, _, t = tokens.shape
    model = torch.arange(m, device=tokens.device)[:, None, None]
    X = w["tok"][model, tokens] + w["pos"][:, None, :t]  # (M, N, T, d): model m looks up its own embedding
    maps = []
    for p in w["layers"]:  # pre-norm blocks with residual connections
        update, A = self_attention(layer_norm(X, p["g1"], p["b1"]), p)
        X = X + update
        X = X + mlp(layer_norm(X, p["g2"], p["b2"]), p)
        maps.append(A)
    Xf = layer_norm(X, w["gf"], w["bf"])
    logits = torch.einsum("mnic,mcv->mniv", Xf, w["head"]) + w["bias"][:, None, None]
    return logits, torch.stack(maps, dim=2)


def masked_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Per-model mean −log p of the trained targets (nats).  logits (M, N, T, V), targets, mask (M, N, T) → (M,)."""
    nll = -torch.gather(torch.log_softmax(logits, -1), -1, targets[..., None])[..., 0]
    return (nll * mask).sum((1, 2)) / mask.sum((1, 2))


def bit_choice(logits: torch.Tensor) -> torch.Tensor:
    """The model's guess when a bit is due: the more likely of the tokens 0 and 1.  (..., V) → (...,) 0/1."""
    return logits[..., :2].argmax(-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/chain_of_thought/tensors.py
"""Build the concrete tensors: bit strings, the two text formats as token ids, and M stacked transformers.

Vocabulary (5 tokens): 0, 1, ":", "=", "_" (blank: padding, and the hidden scratchpad cells of experiment c).

    direct   b_1 … b_n = p                      length n + 2
    CoT      b_1 … b_n : s_1 … s_n = p          length 2n + 3,   s_k = b_1 ⊕ … ⊕ b_k,  p = s_n

Every sequence is padded with blanks to one context length T, so direct and CoT models of every n share one
tensor (M, N, T) and train in one batched run. The weights of M independent models are stacked along a leading
model axis m (no nn.Module, no vmap): every product in equation.py is an explicit einsum over that axis.
"""

import math

import torch

from . import equation

VOCAB = ["0", "1", ":", "=", "_"]
ZERO, ONE, COLON, EQ, BLANK = range(5)


def seq_len(n: int, cot: bool) -> int:
    return 2 * n + 3 if cot else n + 2


def make_batch(bits: torch.Tensor, n: torch.Tensor, cot: torch.Tensor, ctx: int):
    """bits (N, n_max) 0/1, n (N,) string lengths, cot (N,) bool format → tokens, targets, loss mask, each (N, T).

    Built by position arithmetic, no loop over rows: position j holds b_{j+1} for j < n, then the format's tokens.
    targets = tokens shifted one left (next-token prediction). The mask selects what is trained: the answer p
    (direct) or the scratchpad s_1 … s_n, "=" and p (CoT).
    """
    n_max = bits.shape[1]
    j = torch.arange(ctx, device=bits.device)[None, :]  # (1, T) position
    nn = n[:, None]  # (N, 1)
    s = equation.running_parity(bits)  # (N, n_max) prefix parities (cells past n are never read)
    p = s.gather(1, (nn - 1).clamp(min=0))  # (N, 1) answer = last running parity
    bit_at = bits.gather(1, j.clamp(max=n_max - 1).expand(len(n), -1))  # (N, T) b_{j+1}
    s_at = s.gather(1, (j - nn - 1).clamp(0, n_max - 1))  # (N, T) s_{j-n} on the scratchpad cells
    blank = torch.full_like(bit_at, BLANK)
    direct = torch.where(j == nn + 1, p, blank)
    direct = torch.where(j == nn, torch.full_like(blank, EQ), direct)
    chain = torch.where(j == 2 * nn + 2, p, blank)
    chain = torch.where(j == 2 * nn + 1, torch.full_like(blank, EQ), chain)
    chain = torch.where((j > nn) & (j <= 2 * nn), s_at, chain)
    chain = torch.where(j == nn, torch.full_like(blank, COLON), chain)
    tokens = torch.where(j < nn, bit_at, torch.where(cot[:, None], chain, direct))
    targets = torch.cat([tokens[:, 1:], torch.full_like(tokens[:, :1], BLANK)], dim=1)
    mask = torch.where(cot[:, None], (j >= nn) & (j <= 2 * nn + 1), j == nn)
    return tokens, targets, mask


def int_to_bits(ints: torch.Tensor, n_max: int) -> torch.Tensor:
    """Integers (...,) → their n_max lowest bits (..., n_max), least significant first."""
    return (ints[..., None] >> torch.arange(n_max, device=ints.device)) & 1


class StringSplit:
    """A fixed train/test split of the 2^n strings of every length n: a quarter (at most 1024) are test-only.

    Training draws never return a test string, so test accuracy measures generalisation, not memory.
    Below n = 8 there are too few strings to hold any out, and the test set is all 2^n strings.
    """

    def __init__(self, lengths: list[int], seed: int, device: torch.device, n_max: int = 0):
        gen = torch.Generator().manual_seed(seed)
        self.n_max = max(lengths + [n_max])
        self.offset = torch.zeros(self.n_max + 2, dtype=torch.long)
        self.offset[1:] = torch.cumsum(2 ** torch.arange(self.n_max + 1), 0)  # table of length n starts here
        is_test = torch.zeros(int(self.offset[-1]), dtype=torch.bool)
        self.test, self.held_out = {}, {}
        for n in range(1, self.n_max + 1):  # a few small tables, one per length
            self.held_out[n] = n in lengths and 2**n >= 256
            if self.held_out[n]:
                pick = torch.randperm(2**n, generator=gen)[: min(1024, 2**n // 4)]
                is_test[self.offset[n] + pick] = True
            else:  # too few strings to hold any out: test on all of them (this measures fitting, not generalising)
                pick = torch.arange(2**n)
            self.test[n] = pick.to(device)
        self.is_test = is_test.to(device)
        self.offset = self.offset.to(device)

    def sample_train(self, n: torch.Tensor, gen: torch.Generator) -> torch.Tensor:
        """Random training strings of lengths n (...,) → bits (..., n_max); test strings are drawn again."""
        u = torch.rand(n.shape, generator=gen, device=n.device)
        ints = (u * 2.0**n).long()
        bad = self.is_test[self.offset[n] + ints]
        while bad.any():  # rejection sampling: a few rounds at most (a quarter of the strings are excluded)
            u = torch.rand(n.shape, generator=gen, device=n.device)
            ints = torch.where(bad, (u * 2.0**n).long(), ints)
            bad = self.is_test[self.offset[n] + ints]
        return int_to_bits(ints, self.n_max) * (torch.arange(self.n_max, device=n.device) < n[..., None])

    def sample_test(self, n: int, size: int, gen: torch.Generator) -> torch.Tensor:
        """size test strings of length n (with repeats when there are fewer) → bits (size, n_max)."""
        pool = self.test[n]
        pick = torch.randint(len(pool), (size,), generator=gen, device=pool.device)
        return int_to_bits(pool[pick], self.n_max)


def init_weights(m: int, v: int, d: int, heads: int, layers: int, ctx: int, seed: int, device: torch.device) -> dict:
    """M stacked GPT models (leading axis m). c = channel (d), h = head, k = channel within a head (d / heads).

    tok (M, V, d), pos (M, T, d) learned embeddings;  per layer: g1, b1 (M, d) LayerNorm; Wq, Wk, Wv (M, H, d, d/H)
    + biases (M, H, d/H); Wo (M, H, d/H, d) + bo (M, d); g2, b2 (M, d); W1 (M, d, 4d) + c1; W2 (M, 4d, d) + c2;
    gf, bf (M, d) final LayerNorm; head (M, d, V) + bias (M, V). Each model gets its own random draw.
    """
    gen = torch.Generator().manual_seed(seed)
    dk = d // heads
    s = 1 / math.sqrt(d)

    def rand(*shape, scale):
        return (scale * torch.randn(m, *shape, generator=gen)).to(device).requires_grad_(True)

    def const(value, *shape):
        return torch.full((m, *shape), float(value), device=device, requires_grad=True)

    w = {"tok": rand(v, d, scale=0.3), "pos": rand(ctx, d, scale=0.3)}
    w["layers"] = [
        {
            "g1": const(1, d),
            "b1": const(0, d),
            "Wq": rand(heads, d, dk, scale=s),
            "bq": const(0, heads, dk),
            "Wk": rand(heads, d, dk, scale=s),
            "bk": const(0, heads, dk),
            "Wv": rand(heads, d, dk, scale=s),
            "bv": const(0, heads, dk),
            "Wo": rand(heads, dk, d, scale=s / math.sqrt(2 * layers)),
            "bo": const(0, d),
            "g2": const(1, d),
            "b2": const(0, d),
            "W1": rand(d, 4 * d, scale=s),
            "c1": const(0, 4 * d),
            "W2": rand(4 * d, d, scale=0.5 * s / math.sqrt(2 * layers)),
            "c2": const(0, d),
        }  # fmt: skip
        for _ in range(layers)
    ]
    w.update({"gf": const(1, d), "bf": const(0, d), "head": rand(d, v, scale=1e-3), "bias": const(0, v)})
    return w


def parameters(w: dict) -> list[torch.Tensor]:
    """All trainable tensors, for the optimiser (each has the model axis first)."""
    top = [w["tok"], w["pos"], w["gf"], w["bf"], w["head"], w["bias"]]
    return top + [t for layer in w["layers"] for t in layer.values()]


def select(w: dict, idx: torch.Tensor | list[int]) -> dict:
    """The sub-stack of models idx (detached), e.g. only the CoT models for an evaluation."""
    idx = torch.as_tensor(idx, device=w["tok"].device)
    out = {k: v.detach()[idx] for k, v in w.items() if k != "layers"}
    out["layers"] = [{k: v.detach()[idx] for k, v in layer.items()} for layer in w["layers"]]
    return out


def parameter_count(v: int, d: int, layers: int, ctx: int) -> int:
    """One model: V d (tokens) + T d (positions) + L (12 d² + 13 d) (blocks) + 2 d (final LayerNorm) + d V + V."""
    return v * d + ctx * d + layers * (12 * d * d + 13 * d) + 2 * d + d * v + v


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/chain_of_thought/solve.py
"""Train direct and chain-of-thought models in one batched run, then measure what each of them learned.

All models (length n × format × seed, plus the length-generalisation models) are stacked along the model axis
of one weight dict, see by `tensors.init_weights`; one AdamW step updates all of them at once, and gradient
clipping is done per model so the models never influence one another.
"""

import math
from dataclasses import dataclass, field

import torch

from problems._common.progress import steps as progress_steps

from . import equation, tensors


@dataclass
class Config:
    lengths: list[int] = field(default_factory=lambda: [4, 6, 8, 10])  # n of experiment (a)
    seeds: int = 2  # independent replicates (different weights and data)
    steps: int = 600
    d: int = 64
    heads: int = 4
    layers: int = 2
    batch: int = 32
    lr: float = 3e-3
    train_max: int = 8  # experiment (b): trained on 1 ≤ n ≤ train_max ...
    gen_lengths: list[int] = field(default_factory=lambda: [4, 8, 12, 16])  # ... and tested on these n
    test_size: int = 256
    curve_size: int = 64


def clip_per_model(params: list[torch.Tensor], max_norm: float) -> None:
    """Scale model m's gradient so that its own norm is at most max_norm (a global clip would couple the models)."""
    sq = sum(p.grad.pow(2).flatten(1).sum(1) for p in params)  # (M,)
    scale = (max_norm / (sq.sqrt() + 1e-6)).clamp(max=1.0)
    for p in params:
        p.grad.mul_(scale.view(-1, *[1] * (p.dim() - 1)))


def overfit(x, y, mask, w, n_steps: int, lr: float) -> torch.Tensor:
    """Adam on one fixed batch per model (x, y, mask (M, N, T)): the per-model loss for every step (steps, M)."""
    opt = torch.optim.Adam(tensors.parameters(w), lr=lr)
    losses = []
    for _ in range(n_steps):
        loss = equation.masked_loss(equation.forward(x, w)[0], y, mask)
        opt.zero_grad()
        loss.sum().backward()
        opt.step()
        losses.append(loss.detach())
    return torch.stack(losses)


def _batch(bits, n, cot, ctx):
    """make_batch for stacked models: bits (M, N, n_max), n (M, N), cot (M,) → three tensors (M, N, T)."""
    m, rows = n.shape
    flat_cot = cot[:, None].expand(m, rows).reshape(-1)
    out = tensors.make_batch(bits.reshape(m * rows, -1), n.reshape(-1), flat_cot, ctx)
    return tuple(t.reshape(m, rows, ctx) for t in out)


def _at(logits: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
    """logits (M, N, T, V) at position pos (M, N) → (M, N, V)."""
    return logits.gather(2, pos[..., None, None].expand(*pos.shape, 1, logits.shape[-1]))[:, :, 0]


@torch.no_grad()
def greedy(w, bits, n, cot, ctx):
    """Let every model write its own scratchpad, then answer.  bits (M, N, n_max), n (M, N), cot (M,).

    CoT models fill s_1 … s_n one token at a time (step k reads the logits at position n + k − 1 and writes
    position n + k), so later cells are their own guesses, never the truth; the oracle cells that are still to
    the right cannot leak because of the causal mask. Returns the tokens (M, N, T) and the answers (M, N).
    """
    tok = _batch(bits, n, cot, ctx)[0]
    for k in range(1, int(n.max()) + 1) if bool(cot.any()) else []:  # autoregression: a loop over steps
        pos = (n + k - 1).clamp(max=ctx - 2)
        guess = equation.bit_choice(_at(equation.forward(tok, w)[0], pos))  # (M, N)
        write = cot[:, None] & (k <= n)
        old = tok.gather(2, (pos + 1)[..., None])[..., 0]
        tok = tok.scatter(2, (pos + 1)[..., None], torch.where(write, guess, old)[..., None])
    return tok, answer(w, tok, n, cot)


@torch.no_grad()
def answer(w, tok, n, cot):
    """The model's p: its bit choice at the "=" token (position 2n + 1 for CoT, n for direct).  → (M, N)"""
    pos = torch.where(cot[:, None], 2 * n + 1, n)
    return equation.bit_choice(_at(equation.forward(tok, w)[0], pos))


def _test_bits(split, ns: list[int], size: int, gen) -> torch.Tensor:
    """Test strings for a stack of models of lengths ns → (M, size, n_max)."""
    return torch.stack([split.sample_test(n, size, gen) for n in ns])


@torch.no_grad()
def chain_right(w, x, y, mask) -> torch.Tensor:
    """Teacher-forced check of every bit the model must write (s_1 … s_n and p, or p alone).  → (M, N) bool.

    If every next-bit guess is right given the true prefix, the model's own greedy chain is exactly the true
    chain, so this equals "the model writes the whole chain and the answer correctly" in a single forward pass.
    """
    due = mask & (y < 2)  # positions where a bit is due (not the "=" token)
    ok = (equation.bit_choice(equation.forward(x, w)[0]) == y) | ~due
    return ok.all(-1)


def run_experiments(cfg: Config, dev: torch.device, n_checkpoints: int = 24, length_gen: bool = True, seed: int = 0):
    """Experiments (a)–(d) from one training run. Returns a dict of CPU tensors (see main.py).

    Models come in groups that share a context length T: one group per n of experiment (a) holding the direct
    and the CoT models of every seed, (2S models stacked on the model axis, the direct strings padded with blanks),
    plus one group for experiment (b). Every step trains all groups and sums their losses: one AdamW step.
    """
    L, S = len(cfg.lengths), cfg.seeds
    n_max = max(cfg.lengths + cfg.gen_lengths + [cfg.train_max])
    V = len(tensors.VOCAB)
    fmt = torch.tensor([False] * S + [True] * S, device=dev)  # (2S,) model order inside a group: direct, then CoT
    split = tensors.StringSplit(sorted(set(cfg.lengths + cfg.gen_lengths)), seed=seed + 100, device=dev, n_max=n_max)
    groups = [{"n": n, "ctx": 2 * n + 3} for n in cfg.lengths]
    groups += [{"n": 0, "ctx": 2 * n_max + 3}] if length_gen else []
    for g_i, g in enumerate(groups):  # n = 0 marks the length-generalisation group (random n ≤ train_max)
        g["w"] = tensors.init_weights(2 * S, V, cfg.d, cfg.heads, cfg.layers, g["ctx"], seed=seed + g_i, device=dev)
    params = [p for g in groups for p in tensors.parameters(g["w"])]
    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda t: min(1.0, (t + 1) / 30) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * t / cfg.steps)))
    )
    gen = torch.Generator(device=dev).manual_seed(seed + 1)
    test_gen = torch.Generator(device=dev).manual_seed(seed + 2)
    curve = []  # fixed held-out strings per group of (a), in both formats
    for g in groups[:L]:
        bits = _test_bits(split, [g["n"]] * 2 * S, cfg.curve_size, test_gen)
        curve.append(_batch(bits, torch.full((2 * S, cfg.curve_size), g["n"], device=dev), fmt, g["ctx"]))
    checkpoints = sorted({round(k * cfg.steps / n_checkpoints) for k in range(n_checkpoints)} | {cfg.steps})
    hist = {"at": [], "acc": [], "loss": []}

    def checkpoint(t):
        hist["at"].append(t)
        hist["acc"].append(torch.stack([chain_right(g["w"], *c).float().mean(-1).cpu() for g, c in zip(groups, curve)]))

    for t in progress_steps(cfg.steps, "training steps"):
        if t in checkpoints:
            checkpoint(t)
        total, losses = 0.0, []
        for g in groups:
            if g["n"]:
                n = torch.full((2 * S, cfg.batch), g["n"], device=dev)
            else:
                n = torch.randint(1, cfg.train_max + 1, (2 * S, cfg.batch), generator=gen, device=dev)
            t_len = g["ctx"] if g["n"] else 2 * cfg.train_max + 3  # (b) trains on the short strings only
            x, y, mask = _batch(split.sample_train(n, gen), n, fmt, t_len)  # never a held-out string
            loss = equation.masked_loss(equation.forward(x, g["w"])[0], y, mask)  # (2S,)
            total = total + loss.sum()
            losses.append(loss.detach().cpu())
        opt.zero_grad()
        total.backward()  # each model's loss only reaches its own weights
        for g in groups:
            clip_per_model(tensors.parameters(g["w"]), 1.0)
        opt.step()
        sched.step()
        hist["loss"].append(torch.stack(losses[:L]))
    checkpoint(cfg.steps)

    res = {"cfg": cfg, "M": 2 * S * len(groups), "at": torch.tensor(hist["at"]),
           "curve_acc": torch.stack(hist["acc"]).reshape(-1, L, 2, S),  # (C, L, f, S): whole chain right
           "loss": torch.stack(hist["loss"]).reshape(-1, L, 2, S),  # (steps, L, f, S)
           "held_out": [split.held_out[n] for n in cfg.lengths],
           "params_per_model": [tensors.parameter_count(V, cfg.d, cfg.layers, g["ctx"]) for g in groups]}  # fmt: skip

    # (a) final accuracy, (c) hidden scratchpad steps, oracle chain: one group at a time
    acc = torch.zeros(L, 2, S)
    hidden = torch.zeros(L, S, max(cfg.lengths) + 1)
    oracle = torch.zeros(L, S)
    for li, g in enumerate(groups[:L]):
        n_val, ctx = g["n"], g["ctx"]
        bits = _test_bits(split, [n_val] * 2 * S, cfg.test_size, test_gen)
        nn = torch.full((2 * S, cfg.test_size), n_val, device=dev)
        parity = bits.sum(-1) % 2  # cells past n are 0
        tok, ans = greedy(g["w"], bits, nn, fmt, ctx)
        acc[li] = (ans == parity).float().mean(-1).reshape(2, S).cpu()
        w_cot = tensors.select(g["w"], list(range(S, 2 * S)))
        cot_tok, cot_n, cot_flag = tok[S:], nn[S:], fmt[S:]
        j = torch.arange(ctx, device=dev)
        for k in range(max(cfg.lengths) + 1):  # keep the first k model-written steps, blank out s_{k+1} … s_n
            hidden_tok = torch.where((j > n_val + k) & (j <= 2 * n_val), tensors.BLANK, cot_tok)
            hidden[li, :, k] = (answer(w_cot, hidden_tok, cot_n, cot_flag) == parity[S:]).float().mean(-1).cpu()
        oracle_tok = _batch(bits[S:], cot_n, cot_flag, ctx)[0]  # the true chain fed in
        s_n = equation.running_parity(bits[S:, :, :n_val])[..., -1]
        oracle[li] = (answer(w_cot, oracle_tok, cot_n, cot_flag) == s_n).float().mean(-1).cpu()
    res.update(acc=acc, hidden=hidden, oracle_answer_agrees=oracle)

    # (d) attention of the CoT model (seed 0) at one length: where the query that writes s_k looks
    n_show = 8 if 8 in cfg.lengths else max(cfg.lengths)
    g = groups[cfg.lengths.index(n_show)]
    w1 = tensors.select(g["w"], [S])
    bits = _test_bits(split, [n_show], cfg.test_size, test_gen)
    nn = torch.full((1, cfg.test_size), n_show, device=dev)
    tok, ans = greedy(w1, bits, nn, fmt[S:][:1], g["ctx"])
    with torch.no_grad():
        maps = equation.forward(tok, w1)[1][0]  # (N, layers, H, T, T)
    k = torch.arange(1, n_show + 1, device=dev)
    rows = n_show + k - 1  # the query that predicts s_k sits on s_{k-1} (on ":" for k = 1)
    on_b_k = maps[..., rows, k - 1].mean(0)  # (layers, H, n) weight on b_k at every step k
    on_b = on_b_k.mean(-1)  # (layers, H)
    on_prev = maps[..., rows, rows].mean((0, -1))  # weight on itself, s_{k-1}
    best = divmod(int(on_b.flatten().argmax()), cfg.heads)
    res.update(n_show=n_show, on_b=on_b.cpu(), on_prev=on_prev.cpu(), best_head=best,
               sample_tok=tok[0, 0].cpu(), sample_answer=int(ans[0, 0]),
               sample_parity=int(bits[0, 0].sum() % 2), sample_maps=maps[0].cpu(), on_b_k=on_b_k.cpu())  # fmt: skip

    # (b) length generalisation: models trained on n ≤ train_max, tested on every n of (a)
    if length_gen:
        g = groups[-1]
        gen_acc = torch.zeros(2, S, len(cfg.gen_lengths))
        for li, n_val in enumerate(cfg.gen_lengths):
            bits = _test_bits(split, [n_val] * 2 * S, cfg.test_size, test_gen)
            nn = torch.full((2 * S, cfg.test_size), n_val, device=dev)
            _, ans = greedy(g["w"], bits, nn, fmt, g["ctx"])
            gen_acc[:, :, li] = (ans == bits.sum(-1) % 2).float().mean(-1).reshape(2, S).cpu()
        res["gen_acc"] = gen_acc
    return res


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/chain_of_thought/plot.py
"""Animate the experiment: training curves of direct vs CoT models, the accuracy-vs-n bars filling in, then a
CoT model writing its scratchpad one token at a time next to the attention of the query that writes it.
Also a static figure for experiments (b) and (c) and a CSV table of every accuracy."""

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from problems._common.plotting import ACCENT, export_plot_data
from problems._common.progress import report

from . import tensors

FORMAT_COLOR = {"direct": "#9ca3af", "CoT": ACCENT[0]}  # grey = no scratchpad, blue = chain of thought


def save_gif_shared_palette(fig, update, frames: list[int], path: Path, fps: int = 6) -> Path:
    """Render every frame, then quantise all of them with ONE palette so colours never drift between frames."""
    fig.set_dpi(80)
    shots = []
    for i, f in enumerate(frames):
        update(f)
        fig.canvas.draw()
        shots.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]))
        report(i + 1, len(frames), "frames")
    export_plot_data(fig, path)  # numbers behind the final frame (Results page)
    plt.close(fig)
    sample = shots[:: max(1, len(shots) // 12)]
    strip = Image.new("RGB", (sample[0].width, sample[0].height * len(sample)))
    for k, im in enumerate(sample):
        strip.paste(im, (0, k * im.height))
    palette = strip.quantize(colors=255, method=Image.Quantize.MEDIANCUT)
    frames_p = [im.quantize(palette=palette, dither=Image.Dither.NONE) for im in shots]
    tmp = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frames_p[0].save(tmp, save_all=True, append_images=frames_p[1:], duration=int(1000 / fps), loop=0)
    return tmp.replace(path)


def _smooth(x: np.ndarray, win: int) -> np.ndarray:
    """Running mean along axis 0 (same length, shorter window at the start)."""
    c = np.cumsum(np.concatenate([np.zeros((1, *x.shape[1:])), x]), axis=0)
    i = np.arange(1, len(x) + 1)
    lo = np.maximum(0, i - win)
    return (c[i] - c[lo]) / (i - lo).reshape(-1, *[1] * (x.ndim - 1))


def animate(res: dict, path: Path, hold: int = 2) -> Path:
    cfg = res["cfg"]
    lengths = cfg.lengths
    L = len(lengths)
    at = res["at"].numpy()
    curve = res["curve_acc"].numpy().mean(-1)  # (C, L, 2) whole chain right, mean over seeds
    loss = _smooth(res["loss"].numpy().mean(-1), max(1, cfg.steps // 25))  # (steps, L, 2)
    acc = res["acc"].numpy()  # (L, 2, S) final greedy answer accuracy
    n_show = res["n_show"]
    tok = res["sample_tok"].tolist()
    T = 2 * n_show + 3
    A_all = res["sample_maps"].numpy()[..., :T, :T]  # (layers, H, T, T) on the sample string
    l_best, h_best = res["best_head"]
    A = A_all[l_best, h_best]

    fig = plt.figure(figsize=(11.0, 6.6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.0], width_ratios=[1, 1, 1])
    ax_loss, ax_acc, ax_bar = (fig.add_subplot(gs[0, i]) for i in range(3))
    ax_seq = fig.add_subplot(gs[1, :2])
    ax_att = fig.add_subplot(gs[1, 2])

    steps = np.arange(len(loss))
    loss_lines, acc_lines = {}, {}
    for li in range(L):
        for f in range(2):
            ls = "--" if f == 0 else "-"
            (loss_lines[li, f],) = ax_loss.plot([], [], ls, color=ACCENT[li % len(ACCENT)], lw=1.6)
            (acc_lines[li, f],) = ax_acc.plot([], [], ls, color=ACCENT[li % len(ACCENT)], lw=1.6, marker="o", ms=2.5)
    ax_loss.set_xlim(0, cfg.steps)
    ax_loss.set_ylim(0, max(0.8, float(loss.max()) * 1.05))
    ax_loss.set_xlabel("training step (steps)")
    ax_loss.set_ylabel("training loss (nats per trained token)")
    ax_loss.set_title("loss: dashed direct, solid CoT", fontsize=9)
    ax_acc.set_xlim(0, cfg.steps)
    ax_acc.set_ylim(-0.03, 1.03)
    ax_acc.axhline(0.5, color="#6b7280", lw=0.8, ls=":")
    ax_acc.set_xlabel("training step (steps)")
    ax_acc.set_ylabel("whole chain + answer right (fraction)")
    ax_acc.set_title("test strings, mean of seeds", fontsize=9)
    for li, n in enumerate(lengths):
        ax_acc.plot([], [], color=ACCENT[li % len(ACCENT)], lw=1.6, label=f"$n = {n}$")
    ax_acc.legend(fontsize=7, loc="center right")

    x = np.arange(L)
    width = 0.38
    bars = {}
    dots = {}
    for f, name in enumerate(["direct", "CoT"]):
        bars[f] = ax_bar.bar(x + (f - 0.5) * width, np.zeros(L), width * 0.92, color=FORMAT_COLOR[name], label=name)
        (dots[f],) = ax_bar.plot([], [], "o", color="#111827", ms=3.5, ls="none")
    ax_bar.axhline(0.5, color="#6b7280", lw=0.8, ls=":", label="chance")
    ax_bar.set_xticks(x, [f"{n}" + ("" if res["held_out"][i] else "*") for i, n in enumerate(lengths)])
    ax_bar.set_xlim(-0.6, L - 0.4)
    ax_bar.set_ylim(0, 1.25)
    ax_bar.set_xlabel("string length $n$ (bits)")
    ax_bar.set_ylabel("answer accuracy (fraction)")
    ax_bar.legend(fontsize=7, loc="upper center", ncol=3)
    bar_title = ax_bar.set_title("", fontsize=9)

    labels = [tensors.VOCAB[t] for t in tok[:T]]
    labels[-1] = str(res["sample_answer"])  # the model's own answer, not the true one
    im = ax_att.imshow(A, cmap=plt.get_cmap("Purples", 10), vmin=0, vmax=1)
    ax_att.set_xticks(range(T), labels, fontsize=6, family="monospace")
    ax_att.set_yticks(range(T), labels, fontsize=6, family="monospace")
    ax_att.tick_params(length=0, pad=1)
    ax_att.grid(False)
    ax_att.set_xlabel("key position $j$ (token)", fontsize=8)
    ax_att.set_ylabel("query position $i$ (token)", fontsize=8)
    cb = fig.colorbar(im, ax=ax_att, fraction=0.046, pad=0.03)
    cb.set_label("attention weight $A_{ij}$ (fraction)", fontsize=7)
    cb.ax.tick_params(labelsize=6)
    row_box = plt.Rectangle((-0.5, n_show - 0.5), T, 1, fill=False, ec=ACCENT[1], lw=1.4)
    key_box = plt.Rectangle((-0.5, n_show - 0.5), 1, 1, fill=False, ec=ACCENT[4], lw=1.6)
    ax_att.add_patch(row_box)
    ax_att.add_patch(key_box)
    att_title = ax_att.set_title("", fontsize=8)

    ax_seq.axis("off")
    ax_seq.set_xlim(0, 1)
    ax_seq.set_ylim(0, 1)
    cell = 0.92 / T
    boxes, texts = [], []
    for j in range(T):
        xj = 0.04 + j * cell
        r = plt.Rectangle((xj, 0.46), cell * 0.9, 0.2, fc="white", ec="#374151", lw=0.8)
        ax_seq.add_patch(r)
        boxes.append(r)
        texts.append(ax_seq.text(xj + cell * 0.45, 0.56, "", ha="center", va="center", family="monospace", fontsize=10))
    seq_title = ax_seq.text(0.0, 0.95, "", fontsize=9, va="top")
    seq_note = ax_seq.text(0.0, 0.3, "", fontsize=8.5, va="top", color="#374151")

    n_train = len(at)
    n_write = n_show + 2  # write s_1 … s_n, then the answer, then a last view
    frames = list(range((n_train + n_write) * hold))

    def show_sequence(k_written: int):
        """Bits and ":" always; s_1 … s_k written by the model; "=" and the answer once k = n + 1."""
        for j in range(T):
            visible = j <= n_show or (n_show < j <= n_show + k_written) or (k_written > n_show and j >= 2 * n_show + 1)
            texts[j].set_text(labels[j] if visible else "")
            scratch = n_show < j <= 2 * n_show
            fill = "#dbeafe" if scratch and visible else ("#f3f4f6" if j == 2 * n_show + 2 else "white")
            current = scratch and j == n_show + k_written and k_written <= n_show  # the cell being written
            boxes[j].set_facecolor(fill)
            boxes[j].set_edgecolor(ACCENT[1] if current else "#374151")

    def update(frame: int):
        c = frame // hold
        ci = min(c, n_train - 1)
        seen = steps <= at[ci]
        for (li, f), line in loss_lines.items():
            line.set_data(steps[seen], loss[seen, li, f])
        for (li, f), line in acc_lines.items():
            line.set_data(at[: ci + 1], curve[: ci + 1, li, f])
        final = c >= n_train - 1
        for f in range(2):
            heights = acc[:, f].mean(-1) if final else curve[ci, :, f]
            for li, b in enumerate(bars[f]):
                b.set_height(heights[li])
            if final:
                xs = np.repeat(x + (f - 0.5) * width, acc.shape[-1])
                dots[f].set_data(xs, acc[:, f].reshape(-1))
        bar_title.set_text("final answer accuracy, dots = seeds" if final else f"step {at[ci]}: whole chain right")
        if c < n_train:
            show_sequence(0)
            seq_title.set_text(f"CoT model (n = {n_show}) after training: the scratchpad it will write")
            seq_note.set_text("")
            k = 1
        else:
            k = min(c - n_train + 1, n_show + 1)
            show_sequence(k)
            seq_title.set_text("CoT model writes the running parity $s_k = s_{k-1} \\oplus b_k$ one token at a time")
            if k <= n_show:
                seq_note.set_text(f"step {k}: reads $b_{{{k}}}$ = {labels[k - 1]} and $s_{{{k - 1}}}$ → writes "
                                  f"$s_{{{k}}}$ = {labels[n_show + k]}")  # fmt: skip
            else:
                truth = res["sample_parity"]
                seq_note.set_text(f"answer p = {res['sample_answer']} (true parity {truth}): "
                                  "copied from the last scratchpad cell $s_n$")  # fmt: skip
        kk = min(k, n_show)
        row = n_show + kk - 1  # the query that writes s_k
        row_box.set_y(row - 0.5)
        key_box.set_x(kk - 1 - 0.5)
        key_box.set_y(row - 0.5)
        lh = np.unravel_index(A_all[:, :, row, kk - 1].argmax(), A_all.shape[:2])  # head that looks at b_k most
        im.set_data(A_all[lh])
        att_title.set_text(f"layer {lh[0] + 1}, head {lh[1] + 1}: query writing $s_{{{kk}}}$ (red row),\n"
                           f"key $b_{{{kk}}}$ (orange): $A$ = {A_all[lh][row, kk - 1]:.2f}")  # fmt: skip

    fig.tight_layout(h_pad=2.6)
    return save_gif_shared_palette(fig, update, frames, path, fps=4)


def experiments_png(res: dict, path: Path) -> Path:
    """(b) length generalisation and (c) accuracy against the number of written scratchpad steps."""
    cfg = res["cfg"]
    fig, (ax_b, ax_c) = plt.subplots(1, 2, figsize=(10, 3.8), dpi=90)
    if "gen_acc" in res:
        g = res["gen_acc"].numpy()  # (2, S, Lg)
        x = np.arange(len(cfg.gen_lengths))
        for f, name in enumerate(["direct", "CoT"]):
            xs = x + (f - 0.5) * 0.38
            ax_b.bar(xs, g[f].mean(0), 0.35, color=FORMAT_COLOR[name], label=name)
            ax_b.plot(np.repeat(xs, g.shape[1]).reshape(-1, g.shape[1]).T, g[f], "o", color="#111827", ms=3)
        ax_b.set_xticks(x, [str(n) for n in cfg.gen_lengths])
        inside = [i for i, n in enumerate(cfg.gen_lengths) if n <= cfg.train_max]
        if inside:
            label = f"trained on $n \\leq {cfg.train_max}$"
            ax_b.axvspan(-0.5, max(inside) + 0.5, color="#f3f4f6", zorder=0, label=label)
        ax_b.set_xlim(-0.5, len(x) - 0.5)
    ax_b.axhline(0.5, color="#6b7280", lw=0.8, ls=":")
    ax_b.set_ylim(0, 1.05)
    ax_b.set_xlabel("test string length $n$ (bits)")
    ax_b.set_ylabel("answer accuracy (fraction)")
    ax_b.set_title("(b) length generalisation, dots = seeds", fontsize=10)
    ax_b.legend(fontsize=7, loc="upper right")
    n_big = max(cfg.lengths)
    for li, n in enumerate(cfg.lengths):
        h = res["hidden"][li, :, : n + 1].numpy().mean(0)
        ax_c.plot(np.arange(n + 1), h, "o-", color=ACCENT[li % len(ACCENT)], ms=3.5, lw=1.5, label=f"$n = {n}$")
    ax_c.axhline(0.5, color="#6b7280", lw=0.8, ls=":")
    ax_c.set_xlim(-0.3, n_big + 0.3)
    ax_c.set_ylim(0, 1.05)
    ax_c.set_xlabel("scratchpad steps kept $k$ (tokens)")
    ax_c.set_ylabel("CoT answer accuracy (fraction)")
    ax_c.set_title("(c) later steps blanked out at test time", fontsize=10)
    ax_c.legend(fontsize=7, loc="upper left")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    return path


def write_table(res: dict, path: Path) -> Path:
    """Every final accuracy, one row per (experiment, n, format, seed)."""
    cfg = res["cfg"]
    rows = [["experiment", "n", "format", "seed", "accuracy"]]
    for li, n in enumerate(cfg.lengths):
        for f, name in enumerate(["direct", "CoT"]):
            for s in range(cfg.seeds):
                rows.append(["a: fixed n", n, name, s, f"{res['acc'][li, f, s]:.4f}"])
    if "gen_acc" in res:
        for li, n in enumerate(cfg.gen_lengths):
            for f, name in enumerate(["direct", "CoT"]):
                for s in range(cfg.seeds):
                    rows.append([f"b: trained n<={cfg.train_max}", n, name, s, f"{res['gen_acc'][f, s, li]:.4f}"])
    with open(path, "w", newline="", encoding="utf-8") as fh:
        csv.writer(fh).writerows(rows)
    return path


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'steps': 400,
    'lengths': '4,6,8,10',
    'seeds': 2,
    'd_model': 64,
    'heads': 4,
    'layers': 2,
    'batch': 32,
    'lr': 0.003
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.chain_of_thought")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.chain_of_thought.main")
problem_dir = Path("problems/ai/chain_of_thought")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
